# Playground — Geração *instruction-only* (baseline)
## Assunto: Gestão do Desempenho

Condição de comparação para a geração fundamentada em contexto
(`geracao_mcq_petroles.ipynb`). Aqui o LLM **não** recebe trechos do corpus:
o placeholder `{content}` do prompt v13 é preenchido apenas com uma
**descrição do assunto** (instrução temática). Todo o restante é mantido
constante — prompt, modelo, nº de alternativas, validação estrutural e QC —
de modo que a única variável experimental seja o **condicionamento**:

| Condição | `{content}` | Fonte do conhecimento |
|---|---|---|
| *Context-grounded* | trecho do corpus Petrolês | documento recuperado |
| *Instruction-only* | descrição do assunto | conhecimento paramétrico do LLM |

Como não há variação de trecho entre chamadas, questões duplicadas entre
lotes são esperadas; a **taxa de duplicação** é registrada como métrica
comparativa (tende a ser maior nesta condição — achado relevante para o
artigo).

In [ ]:
import os, re, json, random, hashlib, unicodedata, datetime
from pathlib import Path

import numpy as np
import pandas as pd

SEED = 42
random.seed(SEED); np.random.seed(SEED)

BASE_DIR = Path(".")
OUT_DIR = BASE_DIR / "mcq_output"
OUT_DIR.mkdir(exist_ok=True)

TOPIC_ID = "gestao_desempenho"
TOPIC_NAME = "Gestão do Desempenho"

# Descrição do assunto usada como {content} (condição instruction-only)
TOPIC_CONTENT = """\
As questões devem abordar aspectos de gestão e desempenho de FPSOs, incluindo
planejamento e controle operacional, gestão de indicadores e metas, eficiência e
disponibilidade dos sistemas, confiabilidade dos ativos, gestão de riscos, custos,
perdas de produção, desempenho das equipes e processos de melhoria contínua ao
longo do ciclo de vida da unidade."""

# Geração
N_QUESTOES_ALVO = 30
Q_PER_CALL = 10          # questões por chamada (lotes independentes)
N_ALTERNATIVES = 4
MAX_RETRIES = 2
MAX_CALLS = 8            # teto de chamadas (folga p/ duplicatas)

LLM_BACKEND = "mock"     # "anthropic" | "openai" | "ollama" | "transformers" | "mock"
LLM_MODEL = {
    "anthropic": "claude-sonnet-4-5",
    "openai": "gpt-4o",
    "ollama": "qwen3.6:27b",
    "transformers": "Qwen/Qwen3.6-27B",  # ou "Qwen/Qwen3.6-35B-A3B" (MoE, + rápido)
    "mock": "mock",
}[LLM_BACKEND]

from prompt_mcq_generation import PORTUGUESE_V13
PROMPT = PORTUGUESE_V13
print(f"Prompt: {PROMPT.version} | Backend: {LLM_BACKEND} ({LLM_MODEL})")
print(f"Condição: instruction-only | Assunto: {TOPIC_NAME}")

## Backends de LLM (idênticos ao pipeline principal)

In [ ]:
class LLMClient:
    def generate(self, system: str, user: str) -> str:
        raise NotImplementedError


class AnthropicClient(LLMClient):
    def __init__(self, model):
        import anthropic
        self.client = anthropic.Anthropic()  # ANTHROPIC_API_KEY no ambiente
        self.model = model
    def generate(self, system, user):
        r = self.client.messages.create(model=self.model, max_tokens=8192,
                                        system=system,
                                        messages=[{"role": "user",
                                                   "content": user}])
        return r.content[0].text


class OpenAIClient(LLMClient):
    def __init__(self, model):
        from openai import OpenAI
        self.client = OpenAI()  # OPENAI_API_KEY no ambiente
        self.model = model
    def generate(self, system, user):
        r = self.client.chat.completions.create(
            model=self.model,
            messages=[{"role": "system", "content": system},
                      {"role": "user", "content": user}])
        return r.choices[0].message.content


class OllamaClient(LLMClient):
    def __init__(self, model, host="http://localhost:11434"):
        self.model, self.host = model, host
    def generate(self, system, user):
        import urllib.request
        req = urllib.request.Request(
            f"{self.host}/api/chat", method="POST",
            headers={"Content-Type": "application/json"},
            data=json.dumps({"model": self.model, "stream": False,
                             "messages": [
                                 {"role": "system", "content": system},
                                 {"role": "user", "content": user}]}).encode())
        with urllib.request.urlopen(req, timeout=600) as resp:
            return json.loads(resp.read())["message"]["content"]


class TransformersClient(LLMClient):
    """Inferência local via Hugging Face Transformers (ex.: H100).

    Carrega o modelo uma única vez em bfloat16. Uso apenas texto (system +
    user); os exemplos multimodais do model card (imagem) não se aplicam aqui.
    """
    def __init__(self, model_id, max_new_tokens=4096, temperature=0.7):
        import torch
        from transformers import AutoTokenizer
        self.max_new_tokens, self.temperature = max_new_tokens, temperature
        self.tok = AutoTokenizer.from_pretrained(model_id)
        try:
            from transformers import AutoModelForCausalLM
            self.model = AutoModelForCausalLM.from_pretrained(
                model_id, torch_dtype=torch.bfloat16, device_map="auto")
        except Exception:  # model cards multimodais (Qwen3.6 omni)
            from transformers import AutoModelForMultimodalLM, AutoProcessor
            self.tok = AutoProcessor.from_pretrained(model_id)
            self.model = AutoModelForMultimodalLM.from_pretrained(
                model_id, torch_dtype=torch.bfloat16, device_map="auto")
    def generate(self, system, user):
        messages = [{"role": "system", "content": system},
                    {"role": "user", "content": user}]
        inputs = self.tok.apply_chat_template(
            messages, add_generation_prompt=True, tokenize=True,
            return_dict=True, return_tensors="pt").to(self.model.device)
        out = self.model.generate(**inputs,
                                  max_new_tokens=self.max_new_tokens,
                                  do_sample=True,
                                  temperature=self.temperature)
        n_in = inputs["input_ids"].shape[-1]
        text = self.tok.decode(out[0][n_in:], skip_special_tokens=True)
        # remove eventual traço de raciocínio (<think>...</think>)
        return re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL)


class MockClient(LLMClient):
    """Dry-run p/ validar o pipeline sem custo de API."""
    _themes = ["indicadores de disponibilidade", "metas de eficiência",
               "confiabilidade de ativos", "gestão de riscos operacionais",
               "custos de manutenção", "perdas de produção",
               "melhoria contínua", "desempenho de equipes",
               "planejamento operacional", "análise crítica de KPIs"]
    _stems = [
        "Em um FPSO, considere um cenário de {th} com restrições de {x}: qual conduta prioriza o desempenho global da unidade?",
        "Durante a análise crítica mensal, a equipe identifica desvio em {th} associado a {x}: qual deve ser a primeira ação de gestão?",
        "Um gestor precisa decidir entre investir em {th} ou mitigar {x}: qual critério de decisão é tecnicamente mais adequado?",
        "Ao estruturar o plano anual da unidade, como o aspecto de {th} deve ser integrado ao tratamento de {x}?",
        "Após um período de queda de desempenho atribuída a {th}, que relação causal com {x} deve ser investigada prioritariamente?",
    ]
    _stress = ["orçamento e prazo", "janela de parada programada",
               "indisponibilidade de sobressalentes", "metas regulatórias",
               "turnos reduzidos de equipe", "contratos de afretamento"]
    def generate(self, system, user):
        qs = []
        for i in range(Q_PER_CALL):
            th = random.choice(self._themes)
            nonce = hashlib.md5(f"{random.random()}".encode()).hexdigest()[:6]
            qs.append({
                "stem": "[MOCK " + nonce + "] " +
                        random.choice(self._stems).format(
                            th=th, x=random.choice(self._stress)),
                "alternatives": [f"Alternativa {c} ({nonce})" for c in "ABCD"],
                "correct_answer_index": random.randrange(4),
                "difficulty": random.choice(["facil", "media", "dificil"]),
                "correct_reason": "Justificativa simulada.",
            })
        return "```json\n" + json.dumps(qs, ensure_ascii=False) + "\n```"


def make_client(backend, model):
    return {"anthropic": AnthropicClient, "openai": OpenAIClient,
            "ollama": OllamaClient, "transformers": TransformersClient,
            "mock": lambda m: MockClient()}[backend](model)


client = make_client(LLM_BACKEND, LLM_MODEL)

## Parsing e validação estrutural (idênticos ao pipeline principal)

In [ ]:
JSON_RE = re.compile(r"\[.*\]", re.DOTALL)

def normalize(text: str) -> str:
    dec = unicodedata.normalize("NFD", text.lower())
    return "".join(c for c in dec if unicodedata.category(c) != "Mn")


def parse_questions(raw: str):
    m = JSON_RE.search(raw)
    if not m:
        raise ValueError("Nenhum JSON encontrado na resposta")
    data = json.loads(m.group(0))
    if not isinstance(data, list):
        raise ValueError("JSON não é uma lista")
    valid = []
    for q in data:
        if not all(k in q for k in ("stem", "alternatives",
                                    "correct_answer_index", "difficulty",
                                    "correct_reason")):
            continue
        if (len(q["alternatives"]) != N_ALTERNATIVES
                or not isinstance(q["correct_answer_index"], int)
                or not 0 <= q["correct_answer_index"] < N_ALTERNATIVES
                or q["difficulty"] not in ("facil", "media", "dificil")
                or len(str(q["stem"]).split()) < 8):
            continue
        valid.append(q)
    return valid


def is_duplicate(stem, seen_token_sets, threshold=0.75):
    toks = set(normalize(stem).split())
    return any(len(toks & s) / max(len(toks | s), 1) >= threshold
               for s in seen_token_sets)

## Geração em lotes independentes

Cada chamada é independente (mesmo prompt, sem histórico), replicando o
protocolo da condição *context-grounded* (que também usa chamadas
independentes, uma por trecho). Duplicatas entre lotes são removidas
on-line e contabilizadas.

In [ ]:
user_prompt = PROMPT.question_template.format(
    n_questions=Q_PER_CALL, n_alternatives=N_ALTERNATIVES,
    content=TOPIC_CONTENT)

questions, seen = [], []
n_calls = n_gen_total = n_dup = 0

while len(questions) < N_QUESTOES_ALVO and n_calls < MAX_CALLS:
    n_calls += 1
    for attempt in range(MAX_RETRIES + 1):
        try:
            raw = client.generate(PROMPT.system_message, user_prompt)
            qs = parse_questions(raw)
            if not qs:
                raise ValueError("Nenhuma questão válida no lote")
            break
        except Exception as e:
            if attempt == MAX_RETRIES:
                print(f"  Lote {n_calls} falhou: {e}")
                qs = []
    n_gen_total += len(qs)
    for q in qs:
        if len(questions) >= N_QUESTOES_ALVO:
            break
        if is_duplicate(q["stem"], seen):
            n_dup += 1
            continue
        seen.append(set(normalize(q["stem"]).split()))
        q.update({
            "topic": TOPIC_ID,
            "topic_name": TOPIC_NAME,
            "generation_mode": "instruction_only",
            "source_chunk_id": None,
            "source_file": None,
            "instruction_content": TOPIC_CONTENT,
            "prompt_version": PROMPT.version,
            "model": f"{LLM_BACKEND}/{LLM_MODEL}",
            "batch": n_calls,
            "generated_at": datetime.datetime.now()
                            .isoformat(timespec="seconds"),
        })
        questions.append(q)
    print(f"Lote {n_calls}: geradas {len(qs)}, acumuladas {len(questions)}, "
          f"duplicatas até agora {n_dup}")

taxa_dup = n_dup / max(n_gen_total, 1)
print(f"\nTotal: {len(questions)} questões | {n_calls} chamadas | "
      f"taxa de duplicação = {taxa_dup:.1%}")

## Controle de qualidade e estatísticas

In [ ]:
def rebalance_positions(questions):
    for i, q in enumerate(questions):
        target = i % N_ALTERNATIVES
        alts = q["alternatives"][:]
        correct = alts.pop(q["correct_answer_index"])
        random.shuffle(alts)
        q["alternatives"] = alts[:target] + [correct] + alts[target:]
        q["correct_answer_index"] = target
    return questions


questions = rebalance_positions(questions)
df_q = pd.DataFrame(questions)

print("Distribuição de dificuldade (alvo ~30/40/30):")
print(df_q["difficulty"].value_counts())
print("\nPosição da resposta correta:")
print(df_q["correct_answer_index"].value_counts().sort_index())

def length_balance(q):
    lens = [len(a) for a in q["alternatives"]]
    lc = lens[q["correct_answer_index"]]
    others = [l for i, l in enumerate(lens) if i != q["correct_answer_index"]]
    return lc / (sum(others) / len(others) + 1e-9)

ratios = df_q.apply(length_balance, axis=1)
print(f"\nRazão de comprimento correta/distratores: média={ratios.mean():.2f}, "
      f"desvio={ratios.std():.2f}")

# Diversidade lexical dos enunciados (métrica comparativa entre condições)
all_toks = [t for s in df_q["stem"] for t in normalize(s).split()]
ttr = len(set(all_toks)) / max(len(all_toks), 1)
print(f"Type-token ratio dos enunciados: {ttr:.3f}")

## Exportação

In [ ]:
jsonl_path = OUT_DIR / f"dataset_mcq_instrucao_{TOPIC_ID}.jsonl"
csv_path = OUT_DIR / f"dataset_mcq_instrucao_{TOPIC_ID}.csv"
with open(jsonl_path, "w", encoding="utf-8") as f:
    for q in questions:
        f.write(json.dumps(q, ensure_ascii=False) + "\n")

flat = df_q.copy()
for i in range(N_ALTERNATIVES):
    flat[f"alternativa_{chr(65+i)}"] = flat["alternatives"].str[i]
flat["gabarito"] = flat["correct_answer_index"].map(lambda i: chr(65 + i))
flat.drop(columns=["alternatives"]).to_csv(csv_path, index=False,
                                           encoding="utf-8-sig")
print(f"Salvos:\n  {jsonl_path}\n  {csv_path}")

## Comparação com a condição *context-grounded* (se disponível)

Carrega o dataset gerado pelo pipeline principal e compara as duas condições
no mesmo assunto. Métricas automáticas; a comparação decisiva para o artigo é
a avaliação humana cega (especialistas julgam questões das duas condições sem
saber a origem).

In [ ]:
ctx_path = OUT_DIR / "dataset_mcq_petroles.jsonl"
if ctx_path.exists():
    df_ctx = pd.DataFrame([json.loads(l) for l in
                           open(ctx_path, encoding="utf-8")])
    df_ctx = df_ctx[df_ctx.topic == TOPIC_ID]
    if len(df_ctx):
        def stats(df, label):
            toks = [t for s in df["stem"] for t in normalize(s).split()]
            return {
                "condição": label,
                "n_questões": len(df),
                "ttr_enunciados": round(len(set(toks)) / max(len(toks), 1), 3),
                "palavras/enunciado": round(
                    df["stem"].str.split().str.len().mean(), 1),
                "%fácil": round((df.difficulty == "facil").mean() * 100),
                "%média": round((df.difficulty == "media").mean() * 100),
                "%difícil": round((df.difficulty == "dificil").mean() * 100),
            }
        comp = pd.DataFrame([stats(df_q, "instruction-only"),
                             stats(df_ctx, "context-grounded")])
        print(comp.to_string(index=False))
    else:
        print("Dataset context-grounded não tem questões deste assunto ainda.")
else:
    print("Rode antes o geracao_mcq_petroles.ipynb para habilitar a comparação.")

## Notas metodológicas para o artigo

- **Variável experimental única**: o conteúdo do placeholder `{content}`
  (trecho recuperado vs. descrição do assunto). Prompt, modelo, temperatura,
  schema e QC idênticos entre condições.
- **Assimetria esperada**: na condição *instruction-only* o conhecimento vem
  dos parâmetros do modelo (risco de alucinação e de viés para tópicos mais
  frequentes no pré-treino); na *context-grounded*, do documento (permite
  verificação contra a fonte).
- **Métricas automáticas comparativas**: taxa de duplicação entre lotes,
  diversidade lexical (TTR), distribuição de dificuldade, balanceamento de
  comprimento.
- **Avaliação decisiva**: julgamento humano cego por especialistas
  (correção do gabarito, plausibilidade dos distratores, especificidade
  técnica, aderência ao assunto), com acordo inter-anotador (ex.: Cohen's κ).